In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier # Cambiato l'algoritmo
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARICAMENTO DATI
print("📂 Caricamento dati in corso...")
# Utilizziamo i file puliti che abbiamo generato precedentemente
X_train = pd.read_csv('dataset/senza_unknown_elaborato_X_train_v2_nuovo_nuovo_nuovo1.csv')
X_test = pd.read_csv('dataset/senza_unknown_elaborato_X_test_v2_nuovo_nuovo_nuovo1.csv')
y_train = pd.read_csv('dataset/y_train_scaled.csv')
y_test = pd.read_csv('dataset/y_test_scaled.csv')

# Assicuriamoci che l'ordine delle colonne sia identico (previene errori di mismatch)
X_test = X_test[X_train.columns]

# 2. CONFIGURAZIONE E ADDESTRAMENTO
print("🚀 Addestramento Random Forest (Bilanciato)...")
# Nota: scale_pos_weight di XGBoost qui si sostituisce con class_weight='balanced'
model_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,                # Profondità controllata per generalizzare meglio
    min_samples_leaf=5,          # Aiuta a ridurre l'influenza di singoli casi isolati
    class_weight='balanced',     # Gestisce automaticamente lo sbilanciamento delle classi
    random_state=42,
    n_jobs=-1                    # Usa tutti i core del processore per velocizzare
)

model_rf.fit(X_train, y_train.values.ravel())
print("✅ Modello addestrato!")

# ---------------------------------------------------------
# 3. VALUTAZIONE CON SOGLIA PERSONALIZZATA (0.78)
# ---------------------------------------------------------
SOGLIA = 0.78
print(f"⚖️ Applicazione soglia decisionale: {SOGLIA}")

# Otteniamo le probabilità per la classe positiva (1)
y_probs = model_rf.predict_proba(X_test)[:, 1]

# Applichiamo la soglia personalizzata
y_pred_custom = (y_probs >= SOGLIA).astype(int)

# Calcolo delle metriche
acc = accuracy_score(y_test, y_pred_custom)
bal_acc = balanced_accuracy_score(y_test, y_pred_custom)

print("\n" + "="*50)
print(f"🎯 RISULTATI RANDOM FOREST CON SOGLIA {SOGLIA}")
print(f"✨ ACCURATEZZA STANDARD: {acc:.2%}")
print(f"⚖️ BALANCED ACCURACY:  {bal_acc:.2%}")
print("="*50)

print("\n📊 REPORT DETTAGLIATO:")
print(classification_report(y_test, y_pred_custom))

# 4. VISUALIZZAZIONE FEATURE IMPORTANCE
plt.figure(figsize=(10, 7))
importanze = pd.Series(model_rf.feature_importances_, index=X_train.columns).sort_values(ascending=True)
importanze.tail(10).plot(kind='barh', color='salmon') # Colore diverso per distinguerlo da XGBoost
plt.title(f'Top 10 Variabili Determinanti (Random Forest)\n(Soglia applicata: {SOGLIA})')
plt.xlabel('Importanza Relativa (Mean Decrease in Impurity)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# SALVATAGGIO FINALE
joblib.dump(model_rf, 'modello_vaers_rf_finale.pkl')
print("💾 Modello Random Forest salvato con successo!")

📂 Caricamento dati in corso...
🚀 Addestramento Random Forest (Bilanciato)...


Successo su dataset/elaborato_X_train_v2_nuovo_nuovo_nuovo1.csv: rimosse ['VAX_DOSE_SERIES_Unknown', 'VAX_DOSE_SERIES_unknown']
File salvato come: dataset/senza_unknown_elaborato_X_train_v2_nuovo_nuovo_nuovo1.csv
Successo su dataset/elaborato_X_test_v2_nuovo_nuovo_nuovo1.csv: rimosse ['VAX_DOSE_SERIES_Unknown', 'VAX_DOSE_SERIES_unknown']
File salvato come: dataset/senza_unknown_elaborato_X_test_v2_nuovo_nuovo_nuovo1.csv
